In [1]:
import pandas as pd
import numpy as np
import requests
import zipfile
import io
import csv
import os
from datetime import timedelta
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, confusion_matrix

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
tweets = pd.read_csv('/content/drive/MyDrive/labeled_tweets.csv')

In [ ]:
tweets.head()

,text,date,stock_label
0,"...For years, I watched one betrayal after ano...",2020-10-24 02:06:10+00:00,dollar
1,RT @ArthurSchwartz: Texas Lt. Gov. Dan Patrick...,2020-11-11 02:44:06+00:00,dollar
2,European Countries are sadly getting clobbered...,2020-11-16 16:11:09+00:00,euro
3,RT @DonaldJTrumpJr: 🚨🚨🚨Hunter Biden Offered $1...,2020-10-16 04:37:02+00:00,dollar
4,$13.9M is heading to New Orleans in @USDOT fun...,2020-08-12 18:43:05+00:00,dollar


In [5]:
print(len(tweets))

2021


In [ ]:
!pip install -q transformers torch scipy

import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np
import pandas as pd
from scipy.special import softmax
from tqdm.notebook import tqdm

# Wczytanie modelu sentiment analysis trenowanego na tweetach
MODEL = "cardiffnlp/twitter-roberta-base-sentiment"
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

# Funkcja przypisująca predykcję na podstawie sentymentu
def predict_effect(text):
    encoded_input = tokenizer(text, return_tensors='pt', truncation=True, padding=True, max_length=128)
    with torch.no_grad():
        output = model(**encoded_input)
    scores = output.logits[0].numpy()
    scores = softmax(scores)

    # Miejsca: [negative, neutral, positive]
    max_idx = np.argmax(scores)
    if max_idx == 0:
        return 'decrease'
    elif max_idx == 1:
        return 'no change'
    else:
        return 'increase'

# Dodanie kolumny 'predicted_effect' do Twojej tabeli 'tweets'
tqdm.pandas(desc="Analiza tweetów")
tweets['predicted_effect'] = tweets['text'].progress_apply(predict_effect)

# Podgląd wyników
print(tweets[['text', 'stock_label', 'predicted_effect']].head(10))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 46.8 MB/s eta 0:00:00


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/747 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Analiza tweetów:   0%|          | 0/2021 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

                                                text stock_label  \
0  ...For years, I watched one betrayal after ano...      dollar   
1  RT @ArthurSchwartz: Texas Lt. Gov. Dan Patrick...      dollar   
2  European Countries are sadly getting clobbered...        euro   
3  RT @DonaldJTrumpJr: 🚨🚨🚨Hunter Biden Offered $1...      dollar   
4  $13.9M is heading to New Orleans in @USDOT fun...      dollar   
5  GREAT news! New government of Sudan, which is ...      dollar   
6  RT @iheartmindy: Dominion Voting System donate...      dollar   
7  RT @AdamMilstein: Fred Trump sold property to ...      dollar   
8  But 2020 is a long way from over! https://t.co...      dollar   
9  RT @WhiteHouse: The Trump Administration stand...      dollar   

  predicted_effect  
0         decrease  
1        no change  
2         decrease  
3        no change  
4         increase  
5         increase  
6        no change  
7        no change  
8        no change  
9        no change  


In [89]:
tweets.to_csv('/content/drive/MyDrive/tweets_with_predictions.csv', index=False)

In [95]:
tweets = pd.read_csv('/content/drive/MyDrive/tweets_with_predictions.csv')

In [92]:
tweets.head()

,text,date,stock_label,predicted_effect,price_now,price_5min,price_10min,price_1h
0,RT @charliekirk11: Facts:\n\nFederal tax dolla...,2020-03-12 10:00:25+00:00,dollar,no change,time\n2020-03-12 10:00:00 1.1242\n2020-03-1...,time\n2020-03-12 10:05:00 1.12392\n2020-03-...,time\n2020-03-12 10:10:00 1.1232\n2020-03-1...,time\n2020-03-12 11:00:00 1.12372\n2020-03-...
1,RT @WhiteHouse: President @realDonaldTrump sig...,2020-03-08 04:04:06+00:00,dollar,no change,1.13499,1.13499,1.13499,1.13499
2,The Fannie and Freddie execs should not get mi...,2011-11-10 15:26:44+00:00,dollar,decrease,1.3568,1.35532,1.35668,1.35761
3,The Democrat Party has given up on counting vo...,2020-02-07 16:31:56+00:00,dollar,decrease,time\n2020-02-07 16:31:00 1.09479\n2020-02-...,time\n2020-02-07 16:36:00 1.0951\n2020-02-0...,time\n2020-02-07 16:41:00 1.09518\n2020-02-...,time\n2020-02-07 17:31:00 1.09472\n2020-02-...
4,RT @SenateGOP: Trade with Mexico and Canada su...,2020-02-02 04:36:20+00:00,dollar,increase,1.10951,1.10951,1.10951,1.10951


In [98]:
from datetime import datetime

# 1. Ścieżki do danych
BASE_PATHS = {
    'euro': r'/content/drive/MyDrive/THESIS/EUR_USD',
    'sp500': r'/content/drive/MyDrive/THESIS/SPX500_USD',
    'dollar': r'/content/drive/MyDrive/THESIS/EUR_USD',  # tymczasowo EUR/USD
}


In [99]:
# 2. Wczytaj tweety
tweets = pd.read_csv('/content/drive/MyDrive/tweets_with_predictions.csv', parse_dates=["date"])

In [100]:
tweets = tweets[
    (tweets['date'].dt.year.isin(range(2009, 2020))) |
    ((tweets['date'].dt.year == 2020) & (tweets['date'].dt.month <= 5) & (tweets['date'].dt.day <= 13))
].copy()


In [101]:
print(f"Liczba wierszy w dataframe tweets: {len(tweets)}")


Liczba wierszy w dataframe tweets: 1743


In [102]:
import os
import pandas as pd

def get_price_info_at_tweet(row):
    label = row['stock_label']
    timestamp = row['date']

    if pd.isna(label) or pd.isna(timestamp):
        return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

    # Zaokrąglona data i składniki do ścieżki
    date = timestamp.replace(second=0, microsecond=0).tz_localize(None)
    year = str(date.year)
    month = f"{date.month:02d}"
    day = f"{date.day:02d}"
    date_str = f"{year}-{month}-{day}"

    # Ścieżka do pliku dziennego
    path = os.path.join(BASE_PATHS[label], year, month, day, f"{date_str}.csv")

    if not os.path.exists(path):
        print(f"Brak pliku z danymi: {path}")
        return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

    try:
        df = pd.read_csv(path, engine='python')
    except Exception as e:
        print(f"Błąd wczytywania {path}: {e}")
        return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

    # Dopasuj kolumnę czasową
    time_col = 'time' if 'time' in df.columns else 'datetime'
    if time_col not in df.columns:
        print(f"Brak kolumny czasowej w {path}")
        return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

    df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
    df = df.set_index(time_col).sort_index()

    # Czas docelowy i przesunięcia
    base_time = date
    time_points = {
        'price_now': base_time,
        'price_5min': base_time + pd.Timedelta(minutes=5),
        'price_10min': base_time + pd.Timedelta(minutes=10),
        'price_1h': base_time + pd.Timedelta(hours=1)
    }
    results = []
    for label, target_time in time_points.items():
        if target_time in df.index:
            match = df.loc[target_time]
            if isinstance(match, pd.DataFrame):
                results.append(match.iloc[0]['close'])
            else:
                results.append(match['close'])
        else:
            later = df[df.index >= target_time]
            if not later.empty:
                results.append(later.iloc[0]['close'])
            else:
                results.append(None)


    # results = []
    # for label, target_time in time_points.items():
    #     if target_time in df.index:
    #         results.append(df.loc[target_time]['close'])
    #     else:
    #         later = df[df.index >= target_time]
    #         if not later.empty:
    #             results.append(later.iloc[0]['close'])  # najbliższa późniejsza cena
    #         else:
    #             results.append(None)

    return pd.Series(results, index=['price_now', 'price_5min', 'price_10min', 'price_1h'])


In [104]:
from tqdm.notebook import tqdm  # jeśli chcesz pasek postępu

# Dodaj puste kolumny (jeśli jeszcze nie istnieją)
for col in ['price_now', 'price_5min', 'price_10min', 'price_1h']:
    if col not in tweets.columns:
        tweets[col] = None

# Przetwarzanie wszystkich tweetów z paskiem postępu
for i, row in tqdm(tweets.iterrows(), total=len(tweets)):
    prices = get_price_info_at_tweet(row)
    for col in prices.index:
        tweets.at[i, col] = prices[col]

# Podgląd pierwszych 20
display(tweets.iloc[:20][['date', 'text', 'stock_label', 'predicted_effect', 'price_now', 'price_5min', 'price_10min', 'price_1h']])


  0%|          | 0/1743 [00:00<?, ?it/s]

Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/05/02/2020-05-02.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/02/08/2020-02-08.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/04/04/2020-04-04.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/04/04/2020-04-04.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/04/04/2020-04-04.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/05/09/2020-05-09.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2019/12/28/2019-12-28.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/04/11/2020-04-11.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/04/11/2020-04-11.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/04/11/2020-04-11.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2018/01/06/2018-01-06.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2012/1

,date,text,stock_label,predicted_effect,price_now,price_5min,price_10min,price_1h
0,2020-03-12 10:00:25+00:00,RT @charliekirk11: Facts:\n\nFederal tax dolla...,dollar,no change,1.1242,1.12392,1.1232,1.12372
1,2020-03-08 04:04:06+00:00,RT @WhiteHouse: President @realDonaldTrump sig...,dollar,no change,1.13499,1.13499,1.13499,1.13499
2,2011-11-10 15:26:44+00:00,The Fannie and Freddie execs should not get mi...,dollar,decrease,1.3568,1.35532,1.35668,1.35761
3,2020-02-07 16:31:56+00:00,The Democrat Party has given up on counting vo...,dollar,decrease,1.09479,1.0951,1.09518,1.09472
4,2020-02-02 04:36:20+00:00,RT @SenateGOP: Trade with Mexico and Canada su...,dollar,increase,1.10951,1.10951,1.10951,1.10951
5,2020-05-02 23:00:39+00:00,"The Democrats are just, as always, looking for...",euro,decrease,None,None,None,None
6,2020-02-08 12:26:04+00:00,RT @RichardGrenell: Now would be a good time t...,dollar,increase,None,None,None,None
7,2020-03-04 03:23:58+00:00,"The biggest loser tonight, by far, is Mini Mik...",dollar,decrease,1.1165,1.11658,1.11654,1.11579
8,2020-01-09 13:46:26+00:00,Breaking News: The Fifth Circuit Court of Appe...,dollar,increase,1.11136,1.1117,1.11161,1.1109
9,2020-01-10 15:08:14+00:00,"“11,000 points gained in the Dow in the 3 year...",dollar,increase,1.11064,1.11088,1.11084,1.11128


In [103]:
for i in range(20):
    row = tweets.iloc[i]
    prices = get_price_info_at_tweet(row)
    for col in prices.index:
        tweets.at[row.name, col] = prices[col]

display(tweets.iloc[:20][['date', 'text', 'stock_label', 'predicted_effect', 'price_now', 'price_5min', 'price_10min', 'price_1h']])

Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/05/02/2020-05-02.csv
Brak pliku z danymi: /content/drive/MyDrive/THESIS/EUR_USD/2020/02/08/2020-02-08.csv


,date,text,stock_label,predicted_effect,price_now,price_5min,price_10min,price_1h
0,2020-03-12 10:00:25+00:00,RT @charliekirk11: Facts:\n\nFederal tax dolla...,dollar,no change,1.1242,1.12392,1.1232,1.12372
1,2020-03-08 04:04:06+00:00,RT @WhiteHouse: President @realDonaldTrump sig...,dollar,no change,1.13499,1.13499,1.13499,1.13499
2,2011-11-10 15:26:44+00:00,The Fannie and Freddie execs should not get mi...,dollar,decrease,1.3568,1.35532,1.35668,1.35761
3,2020-02-07 16:31:56+00:00,The Democrat Party has given up on counting vo...,dollar,decrease,1.09479,1.0951,1.09518,1.09472
4,2020-02-02 04:36:20+00:00,RT @SenateGOP: Trade with Mexico and Canada su...,dollar,increase,1.10951,1.10951,1.10951,1.10951
5,2020-05-02 23:00:39+00:00,"The Democrats are just, as always, looking for...",euro,decrease,None,None,None,None
6,2020-02-08 12:26:04+00:00,RT @RichardGrenell: Now would be a good time t...,dollar,increase,None,None,None,None
7,2020-03-04 03:23:58+00:00,"The biggest loser tonight, by far, is Mini Mik...",dollar,decrease,1.1165,1.11658,1.11654,1.11579
8,2020-01-09 13:46:26+00:00,Breaking News: The Fifth Circuit Court of Appe...,dollar,increase,1.11136,1.1117,1.11161,1.1109
9,2020-01-10 15:08:14+00:00,"“11,000 points gained in the Dow in the 3 year...",dollar,increase,1.11064,1.11088,1.11084,1.11128


In [84]:
for i in range(4):
    row = tweets.iloc[i]
    prices = get_price_info_at_tweet(row)
    for col in prices.index:
        tweets.at[row.name, col] = prices[col]

display(tweets.iloc[:4][['date', 'text', 'stock_label', 'predicted_effect', 'price_now', 'price_5min', 'price_10min', 'price_1h']])


,date,text,stock_label,predicted_effect,price_now,price_5min,price_10min,price_1h
36,2020-03-12 10:00:25+00:00,RT @charliekirk11: Facts:\n\nFederal tax dolla...,dollar,no change,time 2020-03-12 10:00:00 1.1242 2020-03-12 ...,time 2020-03-12 10:05:00 1.12392 2020-03-12...,time 2020-03-12 10:10:00 1.1232 2020-03-12 ...,time 2020-03-12 11:00:00 1.12372 2020-03-12...
37,2020-03-08 04:04:06+00:00,RT @WhiteHouse: President @realDonaldTrump sig...,dollar,no change,1.13499,1.13499,1.13499,1.13499
40,2011-11-10 15:26:44+00:00,The Fannie and Freddie execs should not get mi...,dollar,decrease,1.3568,1.35532,1.35668,1.35761
43,2020-02-07 16:31:56+00:00,The Democrat Party has given up on counting vo...,dollar,decrease,time 2020-02-07 16:31:00 1.09479 2020-02-07...,time 2020-02-07 16:36:00 1.0951 2020-02-07 ...,time 2020-02-07 16:41:00 1.09518 2020-02-07...,time 2020-02-07 17:31:00 1.09472 2020-02-07...


In [20]:
# 5. Zapisz efekt do nowego pliku CSV
tweets.to_csv("/content/drive/MyDrive/tweets_with_market_price.csv", index=False)

In [17]:
tweets = pd.read_csv("/content/drive/MyDrive/tweets_with_market_price.csv")

In [24]:
tweets.head()

,text,date,stock_label,predicted_effect,price_now,price_5min,price_10min,price_1h,change_5min_pct,change_10min_pct
0,RT @charliekirk11: Facts:\n\nFederal tax dolla...,2020-03-12 10:00:25+00:00,dollar,no change,1.12420,1.12392,1.12320,1.12372,-0.0249%,-0.0890%
1,RT @WhiteHouse: President @realDonaldTrump sig...,2020-03-08 04:04:06+00:00,dollar,no change,1.13499,1.13499,1.13499,1.13499,+0.0000%,+0.0000%
2,The Fannie and Freddie execs should not get mi...,2011-11-10 15:26:44+00:00,dollar,decrease,1.35680,1.35532,1.35668,1.35761,-0.1091%,-0.0088%
3,The Democrat Party has given up on counting vo...,2020-02-07 16:31:56+00:00,dollar,decrease,1.09479,1.09510,1.09518,1.09472,+0.0283%,+0.0356%
4,RT @SenateGOP: Trade with Mexico and Canada su...,2020-02-02 04:36:20+00:00,dollar,increase,1.10951,1.10951,1.10951,1.10951,+0.0000%,+0.0000%


In [23]:
tweets['change_5min_pct'] = tweets.apply(
    lambda row: f"{((row['price_5min'] - row['price_now']) / row['price_now']) * 100:+.4f}%"
    if pd.notnull(row['price_5min']) and pd.notnull(row['price_now']) else None,
    axis=1
)

tweets['change_10min_pct'] = tweets.apply(
    lambda row: f"{((row['price_10min'] - row['price_now']) / row['price_now']) * 100:+.4f}%"
    if pd.notnull(row['price_10min']) and pd.notnull(row['price_now']) else None,
    axis=1
)

In [25]:
tweets.to_csv("/content/drive/MyDrive/tweets_with_percent_change_after_5_and_10_min.csv", index=False)

In [28]:
tweets = pd.read_csv("/content/drive/MyDrive/tweets_with_percent_change_after_5_and_10_min.csv")

In [45]:
threshold = 0.05

tweets_sig_5 = tweets[
    tweets['change_5min_pct'].str.replace('%', '', regex=False).astype(float).abs() >= threshold
].copy()



In [47]:
threshold = 0.05
tweets_sig_10 = tweets[
    tweets['change_10min_pct'].str.replace('%', '', regex=False).astype(float).abs() >= threshold
].copy()

In [35]:
tweets_sig_5.head()

,text,date,stock_label,predicted_effect,price_now,price_5min,price_10min,price_1h,change_5min_pct,change_10min_pct
0,RT @charliekirk11: Facts:\n\nFederal tax dolla...,2020-03-12 10:00:25+00:00,dollar,no change,1.12420,1.12392,1.12320,1.12372,-0.0249%,-0.0890%
2,The Fannie and Freddie execs should not get mi...,2011-11-10 15:26:44+00:00,dollar,decrease,1.35680,1.35532,1.35668,1.35761,-0.1091%,-0.0088%
3,The Democrat Party has given up on counting vo...,2020-02-07 16:31:56+00:00,dollar,decrease,1.09479,1.09510,1.09518,1.09472,+0.0283%,+0.0356%
8,Breaking News: The Fifth Circuit Court of Appe...,2020-01-09 13:46:26+00:00,dollar,increase,1.11136,1.11170,1.11161,1.11090,+0.0306%,+0.0225%
9,"“11,000 points gained in the Dow in the 3 year...",2020-01-10 15:08:14+00:00,dollar,increase,1.11064,1.11088,1.11084,1.11128,+0.0216%,+0.0180%


In [46]:
len(tweets_sig_5)


134

In [48]:
len(tweets_sig_10)

252

In [49]:
tweets_sig_5.to_csv("/content/drive/MyDrive/tweets_0.05_change_after_5_", index=False)

In [50]:
tweets_sig_10.to_csv("/content/drive/MyDrive/tweets_0.05_change_after_10_", index=False)

In [41]:
# path = "/content/drive/MyDrive/THESIS/EUR_USD/2009/01/01/2009-01-01.csv"
# try:
#     df_test = pd.read_csv(path, engine='python')
#     print("OK! Plik da się wczytać.")
# except Exception as e:
#     print(f"Plik uszkodzony lub błędny: {e}")


OK! Plik da się wczytać.


In [53]:
# def get_price_info_at_tweet(row):
#     label = row['stock_label']
#     timestamp = row['date']

#     if pd.isna(label) or pd.isna(timestamp):
#         return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

#     year = timestamp.year
#     path = os.path.join(BASE_PATHS[label], f"{year}.csv")

#     if not os.path.exists(path):
#         print(f"Brak danych dla {label} w roku {year}")
#         return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

#     try:
#         df = pd.read_csv(path, engine='python')
#     except Exception as e:
#         print(f"Błąd wczytywania {path}: {e}")
#         return pd.Series([None, None, None, None], index=['price_now', 'price_5min', 'price_10min', 'price_1h'])

#     time_col = 'time' if 'time' in df.columns else 'datetime'
#     df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
#     df = df.set_index(time_col).sort_index()

#     # Ustaw timestamps (bez strefy czasowej)
#     rounded = timestamp.replace(second=0, microsecond=0).tz_localize(None)
#     times = {
#         'price_now': rounded,
#         'price_5min': rounded + pd.Timedelta(minutes=5),
#         'price_10min': rounded + pd.Timedelta(minutes=10),
#         'price_1h': rounded + pd.Timedelta(hours=1)
#     }

#     # Szukaj najbliższej późniejszej wartości
#     result = []
#     for label, t in times.items():
#         if t in df.index:
#             result.append(df.loc[t]['close'])
#         else:
#             later = df[df.index >= t]
#             if not later.empty:
#                 result.append(later.iloc[0]['close'])
#             else:
#                 result.append(None)

#     return pd.Series(result, index=['price_now', 'price_5min', 'price_10min', 'price_1h'])


In [26]:
# # Sprawdź 1 konkretny tweet z 2019 lub 2020 roku
# example = tweets.iloc[0]  # weź pierwszy tweet (możesz podmienić na inny)

# price = get_price_at_tweet(example)

# print("Tweet:")
# print(example[['text', 'stock_label', 'date']])
# print("\nCena rynkowa przypisana do tego tweeta:", price)


Tweet:
text           At my meeting with Jay Powell this morning, I ...
stock_label                                               dollar
date                                   2019-11-19 03:34:12+00:00
Name: 91, dtype: object

Cena rynkowa przypisana do tego tweeta: 1.10744


In [47]:
tweets = tweets.head(5)

In [48]:
tweets.head()

,text,date,stock_label,predicted_effect,price_now,price_5min,price_10min,price_1h,price_at_tweet
40,The Fannie and Freddie execs should not get mi...,2011-11-10 15:26:44+00:00,dollar,decrease,1.35680,1.35532,1.35668,1.35761,None
91,"At my meeting with Jay Powell this morning, I ...",2019-11-19 03:34:12+00:00,dollar,decrease,1.10744,1.10744,1.10745,1.10750,None
92,"Just finished a very good &amp, cordial meetin...",2019-11-18 16:01:42+00:00,dollar,increase,1.10823,1.10826,1.10868,1.10797,None
93,"Republicans &amp, others must remember, the Uk...",2019-11-17 20:08:08+00:00,euro,decrease,1.10528,1.10528,1.10528,1.10528,None
104,I will be signing our 738 Billion Dollar Defen...,2019-12-20 14:19:20+00:00,dollar,increase,NaN,NaN,NaN,NaN,NaN


In [48]:
# def get_price_at_tweet(row):
#     label = row['stock_label']
#     timestamp = row['date']

#     if pd.isna(label) or pd.isna(timestamp):
#         return None

#     year = timestamp.year
#     path = os.path.join(BASE_PATHS[label], f"{year}.csv")

#     if not os.path.exists(path):
#         print(f"Brak danych dla {label} w roku {year}")
#         return None

#     try:
#         df = pd.read_csv(path, engine='python')  # bezpieczny parser
#     except Exception as e:
#         print(f"Błąd wczytywania {path}: {e}")
#         return None

#     # Dopasuj kolumnę z czasem
#     time_col = 'time' if 'time' in df.columns else 'datetime'
#     df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
#     df = df.set_index(time_col)
#     df = df.sort_index()

#     # Zaokrąglij do minuty (usuń sekundy i mikrosekundy)
#     rounded_time = timestamp.replace(second=0, microsecond=0).tz_localize(None)


#     # Znajdź dokładnie tę minutę lub najbliższą późniejszą
#     if rounded_time in df.index:
#         return df.loc[rounded_time]['close']
#     else:
#         later = df[df.index >= rounded_time]
#         if not later.empty:
#             return later.iloc[0]['close']
#         else:
#             return None


In [43]:
# # 3. Funkcja do pobierania ceny z danych rynkowych
# def get_price_at_tweet(row):
#     label = row['stock_label']
#     timestamp = row['date']

#     if pd.isna(label) or pd.isna(timestamp):
#         return None

#     year = timestamp.year
#     path = os.path.join(BASE_PATHS[label], f"{year}.csv")

#     if not os.path.exists(path):
#         print(f"Brak danych dla {label} w roku {year}")
#         return None

#     try:
#         df = pd.read_csv(path)
#     except Exception as e:
#         print(f"Błąd wczytywania {path}: {e}")
#         return None

#     # Dopasuj nazwę kolumny z czasem
#     time_col = 'time' if 'time' in df.columns else 'datetime'

#     df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
#     df = df.set_index(time_col)

#     # Znajdź dokładną minutę
#     rounded_time = timestamp.replace(second=0, microsecond=0)

#     if rounded_time in df.index:
#         return df.loc[rounded_time]['close']
#     else:
#         return None


In [32]:
# def get_price_at_tweet(row):
#     label = row['stock_label']  # np. 'SPX500_USD'
#     timestamp = row['date']     # pd.Timestamp

#     if pd.isna(label) or pd.isna(timestamp):
#         return None

#     # Rozbij datę
#     year = str(timestamp.year)
#     month = f"{timestamp.month:02d}"
#     day = f"{timestamp.day:02d}"
#     date_str = f"{year}-{month}-{day}"

#     # Ścieżka do pliku dziennego
#     path = os.path.join(BASE_PATHS[label], year, month, day, f"{date_str}.csv")

#     if not os.path.exists(path):
#         print(f"Brak pliku z danymi: {path}")
#         return None

#     try:
#         df = pd.read_csv(path)
#     except Exception as e:
#         print(f"Błąd wczytywania {path}: {e}")
#         return None

#     # Dopasuj kolumnę z czasem
#     time_col = 'time' if 'time' in df.columns else 'datetime'

#     if time_col not in df.columns:
#         print(f"Brak kolumny czasowej w {path}")
#         return None

#     # Konwersja czasu i ustawienie jako indeks
#     df[time_col] = pd.to_datetime(df[time_col], errors='coerce')
#     df = df.set_index(time_col)

#     # Zaokrąglij czas do pełnej minuty
#     rounded_time = timestamp.replace(second=0, microsecond=0)

#     # Znajdź dokładny moment
#     if rounded_time in df.index:
#         return df.loc[rounded_time]['close']
#     else:
#         return None
